In [1]:
# ============================================
# 02_upload_to_pinecone.ipynb (FIXED - with file checking)
# Purpose: Generate embeddings and upload to Pinecone
# ============================================

# Cell 1: Import libraries
import json
import numpy as np
import time
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# ============================================
# Cell 2: CHECK IF FILES EXIST FIRST
# ============================================

print("="*60)
print("CHECKING FOR REQUIRED FILES")
print("="*60)

required_files = ['chunks_fixed.json', 'chunks_recursive.json']
missing_files = []

for file in required_files:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024
        print(f"✅ {file} found ({size:.1f} KB)")
    else:
        print(f"❌ {file} NOT FOUND!")
        missing_files.append(file)

if missing_files:
    print(f"\n❌ ERROR: Missing files: {missing_files}")
    print("\nPlease run '01_data_preparation.ipynb' first and make sure it completes successfully.")
    print("The chunk files should be created in Cell 7 of that notebook.")
    raise FileNotFoundError(f"Missing required files: {missing_files}")

CHECKING FOR REQUIRED FILES
✅ chunks_fixed.json found (6625.2 KB)
✅ chunks_recursive.json found (6808.4 KB)


In [3]:
# ============================================
# Cell 3: Load the chunks
# ============================================

print("\n" + "="*60)
print("LOADING CHUNKS")
print("="*60)

with open('chunks_fixed.json', 'r') as f:
    chunks_fixed = json.load(f)
print(f"✅ Loaded {len(chunks_fixed)} fixed-size chunks")

with open('chunks_recursive.json', 'r') as f:
    chunks_recursive = json.load(f)
print(f"✅ Loaded {len(chunks_recursive)} recursive chunks")

print(f"📊 Total chunks to upload: {len(chunks_fixed) + len(chunks_recursive)}")

if len(chunks_fixed) == 0 or len(chunks_recursive) == 0:
    print("\n❌ ERROR: One of the chunk files is empty!")
    print("Please re-run Phase 1 and ensure chunking works correctly.")
    raise ValueError("Empty chunk files")

# Display sample
print(f"\n📝 Sample chunk metadata: {list(chunks_fixed[0].keys())}")
print(f"📝 Sample chunk text (first 100 chars): {chunks_fixed[0]['text'][:100]}...")


LOADING CHUNKS
✅ Loaded 9421 fixed-size chunks
✅ Loaded 12228 recursive chunks
📊 Total chunks to upload: 21649

📝 Sample chunk metadata: ['source_title', 'source_url', 'chunk_id', 'text', 'chunking_strategy', 'chunk_size']
📝 Sample chunk text (first 100 chars): The Indus Valley Civilisation (IVC), also known as the Indus Civilisation or the Harappan Civilisati...


In [4]:
# ============================================
# Cell 4: Initialize embedding model
# ============================================

print("\n" + "="*60)
print("LOADING EMBEDDING MODEL")
print("="*60)

from sentence_transformers import SentenceTransformer

print("Loading embedding model (this may take a moment)...")
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(f"✅ Model loaded! Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

# Test the model
test_text = "Pakistan gained independence in 1947"
test_embedding = embedding_model.encode(test_text)
print(f"✅ Test embedding shape: {test_embedding.shape}")


LOADING EMBEDDING MODEL


d:\IBA\Sem-6\NLP with Deep Learning\Assignment 3\pakistan_history-rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model (this may take a moment)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5588.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded! Embedding dimension: 384
✅ Test embedding shape: (384,)


In [6]:
# ============================================
# Cell 5: Set up Pinecone
# ============================================

print("\n" + "="*60)
print("SETTING UP PINECONE")
print("="*60)

# Get API credentials
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_ENVIRONMENT = os.getenv("PINECONE_ENVIRONMENT")

print(f"Pinecone API Key: {'✅ Loaded' if PINECONE_API_KEY else '❌ Missing!'}")
print(f"Pinecone Environment: {PINECONE_ENVIRONMENT if PINECONE_ENVIRONMENT else '❌ Missing!'}")

if not PINECONE_API_KEY:
    print("\n❌ ERROR: PINECONE_API_KEY not found in .env file!")
    print("Please create a .env file with:")
    print("PINECONE_API_KEY=your_key_here")
    print("PINECONE_ENVIRONMENT=your_environment_here")
    raise ValueError("Missing Pinecone API key")

if not PINECONE_ENVIRONMENT:
    print("\n❌ ERROR: PINECONE_ENVIRONMENT not found in .env file!")
    print("Your environment is shown in your Pinecone dashboard URL")
    print("Example: if URL is 'app.pinecone.io/organizations/.../projects/gcp-starter'")
    print("Then environment is 'gcp-starter'")
    raise ValueError("Missing Pinecone environment")

# Initialize Pinecone
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)
print("✅ Pinecone client initialized")

# Define index name
INDEX_NAME = "pakistan-history-rag"

# Check existing indexes
existing_indexes = pc.list_indexes().names()
print(f"\nExisting indexes: {existing_indexes if existing_indexes else 'None'}")

# Handle existing index
if INDEX_NAME in existing_indexes:
    print(f"\n⚠ Index '{INDEX_NAME}' already exists.")
    response = input(f"Do you want to delete it and create a fresh one? (yes/no): ")
    if response.lower() == 'yes':
        print(f"Deleting {INDEX_NAME}...")
        pc.delete_index(INDEX_NAME)
        print("Waiting for deletion...")
        time.sleep(5)
    else:
        print("Will use existing index. Note: Old data will remain.")
else:
    print(f"\n📦 Creating new index: {INDEX_NAME}")

# Create index if it doesn't exist
if INDEX_NAME not in pc.list_indexes().names():
    try:
        pc.create_index(
            name=INDEX_NAME,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )
        print("✅ Index creation initiated!")
        print("Waiting for index to be ready...")
        time.sleep(30)
        
        # Wait for index to be ready
        max_wait = 60
        waited = 0
        while INDEX_NAME not in pc.list_indexes().names() and waited < max_wait:
            print(f"  Still waiting... ({waited}s)")
            time.sleep(10)
            waited += 10
            
    except Exception as e:
        print(f"❌ Error creating index: {e}")
        raise

# Connect to index
index = pc.Index(INDEX_NAME)
print(f"✅ Connected to index: {INDEX_NAME}")

# Show index stats
stats = index.describe_index_stats()
print(f"📊 Current index stats: {stats['total_vector_count']} vectors")


SETTING UP PINECONE
Pinecone API Key: ✅ Loaded
Pinecone Environment: us-east1-aws
✅ Pinecone client initialized

Existing indexes: ['pakistan-history-rag']

⚠ Index 'pakistan-history-rag' already exists.
Deleting pakistan-history-rag...
Waiting for deletion...
✅ Index creation initiated!
Waiting for index to be ready...
✅ Connected to index: pakistan-history-rag
📊 Current index stats: 0 vectors


In [7]:
# ============================================
# Cell 6: Upload chunks (with progress)
# ============================================

print("\n" + "="*60)
print("UPLOADING CHUNKS TO PINECONE")
print("="*60)

import re
from tqdm.auto import tqdm

def upload_chunks_to_pinecone(chunks, index, strategy_name, batch_size=50):
    """
    Upload chunks to Pinecone
    """
    total = len(chunks)
    print(f"\n📤 Uploading {total} chunks for '{strategy_name}' strategy...")
    
    successful = 0
    failed = 0
    
    # Process in batches
    for i in tqdm(range(0, total, batch_size), desc=f"Uploading {strategy_name}"):
        batch = chunks[i:i+batch_size]
        vectors = []
        
        for chunk in batch:
            try:
                # Generate embedding
                embedding = embedding_model.encode(chunk['text'])
                
                # Create clean ID
                vector_id = f"{strategy_name}_{chunk['source_title']}_{chunk['chunk_id']}"
                vector_id = re.sub(r'[^a-zA-Z0-9_-]', '_', vector_id)
                if len(vector_id) > 512:
                    vector_id = vector_id[:512]
                
                # Prepare metadata
                metadata = {
                    "text": chunk['text'][:30000],
                    "source_title": chunk['source_title'],
                    "source_url": chunk.get('source_url', ''),
                    "chunking_strategy": strategy_name,
                    "chunk_size": chunk['chunk_size']
                }
                
                vectors.append({
                    "id": vector_id,
                    "values": embedding.tolist(),
                    "metadata": metadata
                })
                
            except Exception as e:
                print(f"  Error processing chunk: {e}")
                failed += 1
                continue
        
        # Upload batch
        if vectors:
            try:
                index.upsert(vectors=vectors)
                successful += len(vectors)
            except Exception as e:
                print(f"  Error uploading batch: {e}")
                failed += len(vectors)
    
    print(f"✅ {strategy_name} upload complete: {successful} successful, {failed} failed")
    return successful

# Upload both strategies
fixed_uploaded = upload_chunks_to_pinecone(chunks_fixed, index, "fixed", batch_size=50)
recursive_uploaded = upload_chunks_to_pinecone(chunks_recursive, index, "recursive", batch_size=50)


UPLOADING CHUNKS TO PINECONE

📤 Uploading 9421 chunks for 'fixed' strategy...


Uploading fixed: 100%|██████████| 189/189 [03:57<00:00,  1.26s/it]


✅ fixed upload complete: 9421 successful, 0 failed

📤 Uploading 12228 chunks for 'recursive' strategy...


Uploading recursive: 100%|██████████| 245/245 [04:30<00:00,  1.10s/it]

✅ recursive upload complete: 12228 successful, 0 failed


In [9]:
# ============================================
# Cell 7: Verify upload
# ============================================

print("\n" + "="*60)
print("VERIFYING UPLOAD")
print("="*60)

stats = index.describe_index_stats()
print(f"📊 Total vectors in Pinecone: {stats['total_vector_count']}")
print(f"📊 Expected total: {fixed_uploaded + recursive_uploaded}")

if stats['total_vector_count'] > 0:
    print("\n✅ SUCCESS! Data successfully uploaded to Pinecone!")
    
    # Test search
    print("\n🔍 Testing search with sample query...")
    test_query = "When did Pakistan gain independence?"
    query_embedding = embedding_model.encode(test_query)
    
    results = index.query(
        vector=query_embedding.tolist(),
        top_k=3,
        include_metadata=True
    )
    
    print(f"\nQuery: {test_query}")
    for i, match in enumerate(results['matches'], 1):
        print(f"\nResult {i}:")
        print(f"  Score: {match['score']:.3f}")
        print(f"  Source: {match['metadata'].get('source_title', 'Unknown')}")
        print(f"  Text: {match['metadata'].get('text', '')[:150]}...")
else:
    print("\n❌ ERROR: No vectors uploaded!")


VERIFYING UPLOAD
📊 Total vectors in Pinecone: 21649
📊 Expected total: 21649

✅ SUCCESS! Data successfully uploaded to Pinecone!

🔍 Testing search with sample query...

Query: When did Pakistan gain independence?

Result 1:
  Score: 0.717
  Source: History of Karachi
  Text: == Post-Independence (1947 CE – present) ==


=== Pakistan's capital (1947–1959) ===...

Result 2:
  Score: 0.695
  Source: Constitution of Pakistan of 1956
  Text: Pakistan became independent from the United Kingdom in 1947, but remained a British Dominion, like Canada and Australia, until 1956. Under Section 8 o...

Result 3:
  Score: 0.684
  Source: Chaudhry Rehmat Ali
  Text: Choudhary Rehmat Ali was the personality who started independence movement in the name of Pakistan. He gave idea of two nations in 1915. He formed Pak...


In [10]:
# ============================================
# Cell 8: Save configuration
# ============================================

config = {
    'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
    'embedding_dimension': 384,
    'pinecone_index': INDEX_NAME,
    'total_vectors': stats['total_vector_count'],
    'fixed_chunks': fixed_uploaded,
    'recursive_chunks': recursive_uploaded
}

with open('pinecone_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"\n✅ Configuration saved to 'pinecone_config.json'")
print("\n" + "="*60)
print("✅ PHASE 2 COMPLETE!")
print("="*60)


✅ Configuration saved to 'pinecone_config.json'

✅ PHASE 2 COMPLETE!
